In [1]:
import json
import os
import requests

json_path = os.path.join(os.path.abspath(os.getcwd()), "zutphen-met-labels.json")
output_folder = os.path.join(os.path.abspath(os.getcwd()), "zutphen-zonder-gebouwen-map")
timeout_seconds = 5

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

with open(json_path, 'r') as f:
    data = json.load(f)

print(f"Start met downloaden van {len(data)} afbeeldingen...")

for index, item in enumerate(data):
    url = item.get("URL")
    if not url:
        print(f"Skipping index {index}: Geen URL gevonden.")
        continue
    try:
        filename = f"image_{index}.jpg"
        save_path = os.path.join(output_folder, filename)
        response = requests.get(url, timeout=timeout_seconds)
        if response.status_code == 200:
            with open(save_path, 'wb') as img_file:
                img_file.write(response.content)
            print(f"Succes: {filename} opgeslagen.")
        else:
            print(f"Fout bij {url}: Status {response.status_code}")
    except Exception as e:
        print(f"Fout bij downloaden van {url}: {e}")


Start met downloaden van 625 afbeeldingen...
Fout bij downloaden van http://localhost/mapserver?SERVICE=WMS&VERSION=1.3.0&REQUEST=GetMap&BBOX=718124.3832041401183%2C5781604.5745665906%2C718504.381002129172%2C5781984.572359534912&CRS=EPSG%3A25831&WIDTH=512&HEIGHT=512&LAYERS=brtachtergrondkaart&STYLES=&FORMAT=image%2Fpng&DPI=96&MAP_RESOLUTION=96&FORMAT_OPTIONS=dpi%3A96&TRANSPARENT=TRUE: HTTPConnectionPool(host='localhost', port=80): Max retries exceeded with url: /mapserver?SERVICE=WMS&VERSION=1.3.0&REQUEST=GetMap&BBOX=718124.3832041401183%2C5781604.5745665906%2C718504.381002129172%2C5781984.572359534912&CRS=EPSG%3A25831&WIDTH=512&HEIGHT=512&LAYERS=brtachtergrondkaart&STYLES=&FORMAT=image%2Fpng&DPI=96&MAP_RESOLUTION=96&FORMAT_OPTIONS=dpi%3A96&TRANSPARENT=TRUE (Caused by NewConnectionError("HTTPConnection(host='localhost', port=80): Failed to establish a new connection: [Errno 111] Connection refused"))
Fout bij downloaden van http://localhost/mapserver?SERVICE=WMS&VERSION=1.3.0&REQUEST=G

In [2]:
from PIL import Image
import numpy as np

input_image_array = np.empty([625,512,512,3], dtype=np.int16)
target_image_array = np.empty([625,512,512,3], dtype=np.int16)

for i in range(625):
    image_path1 = os.path.join("zutphen-zonder-labels-map", f"image_{i}.jpg")
    image1 = Image.open(image_path1)
    image_array1 = np.array(image1)
    input_image_array[i] = np.delete(np.delete(np.delete(image_array1, np.s_[512::], 0),np.s_[512::], 1),np.s_[3::], 2)

    image_path2 = os.path.join("zutphen-met-alleen-labels-map", f"image_{i}.jpg")
    image2 = Image.open(image_path2)
    image_array2 = np.array(image2)
    target_image_array[i] = np.delete(np.delete(np.delete(image_array2, np.s_[512::], 0),np.s_[512::], 1),np.s_[3::], 2)

In [3]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(input_image_array, target_image_array, test_size=0.2, random_state=42)
x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)
y_test = np.array(y_test)

In [4]:
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, UpSampling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


def create_model(img_width, img_height):

    x = Input(shape=(img_width, img_height, 3))

    # Encoder - compresses the input into a latent representation
    e_conv1 = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    pool1 = MaxPooling2D((2, 2), padding='same')(e_conv1)
    batchnorm_1 = BatchNormalization()(pool1)

    e_conv2 = Conv2D(32, (3, 3), activation='relu', padding='same')(batchnorm_1)
    pool2 = MaxPooling2D((2, 2), padding='same')(e_conv2)
    batchnorm_2 = BatchNormalization()(pool2)

    e_conv3 = Conv2D(16, (3, 3), activation='relu', padding='same')(batchnorm_2)
    h = MaxPooling2D((2, 2), padding='same')(e_conv3)

    # Decoder - reconstructs the input from a latent representation
    d_conv1 = Conv2D(64, (3, 3), activation='relu', padding='same')(h)
    up1 = UpSampling2D((2, 2))(d_conv1)

    d_conv2 = Conv2D(32, (3, 3), activation='relu', padding='same')(up1)
    up2 = UpSampling2D((2, 2))(d_conv2)

    d_conv3 = Conv2D(16, (3, 3), activation='relu', padding="same")(up2)
    up3 = UpSampling2D((2, 2))(d_conv3)

    r = Conv2D(3, (3, 3), activation='sigmoid', padding='same')(up3)

    model = Model(x, r)
    model.compile(optimizer=Adam(learning_rate=0.0005), loss='mse')
    return model

2026-02-17 11:57:23.652993: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-17 11:57:23.674621: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-17 11:57:24.300637: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
